In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# load data
reviews = pd.read_csv('/Users/eggy/Downloads/reviews.csv.gz', compression='gzip')
listings_clean = pd.read_csv('data/listings_clean.csv')

print(f'Reviews: {reviews.shape}')
print(f'Listings: {listings_clean.shape}')
print(reviews.columns.tolist())

Reviews: (977459, 6)
Listings: (36613, 23)
['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments']


In [8]:
# basic stats
print('Reviews per listing:')
print(reviews.groupby('listing_id')['id'].count().describe())

print('\nDate range:')
print(f'Earliest review: {reviews["date"].min()}')
print(f'Latest review: {reviews["date"].max()}')

print('\nMissing comments:')
print(reviews['comments'].isna().sum())

print('\nSample reviews:')
print(reviews['comments'].dropna().sample(5, random_state=42).tolist())

Reviews per listing:
count    25093.000000
mean        38.953453
std         79.458967
min          1.000000
25%          3.000000
50%         11.000000
75%         42.000000
max       3518.000000
Name: id, dtype: float64

Date range:
Earliest review: 2009-05-25
Latest review: 2025-08-02

Missing comments:
260

Sample reviews:
['Fantastic to get to live in Williamsburg at A and A’s place! We loved the surroundings and our hosts. They were sweet, nice, helpful, inviting people! Even the neighbours were nice. We had good talks on the wonderful patio.<br/>The room and private bathrooms was fine and clean and with a good WiFi ;-) The bed was a little soft... but equipped with tons of good pillows :-) <br/>It’s SO easy (short walk) to get to the subway - and only a short train ride to Manhattan. We enjoyed Graham Avenue’s shops, restaurants, a great barber shop - and the quiet neighbourhood. ', 'We loved staying here!  It was beyond what we expected, clean, comfortable and so well done!  Lo

In [9]:
# drop missing comments
reviews = reviews.dropna(subset=['comments'])

# clean HTML tags
reviews['comments'] = reviews['comments'].str.replace('<br/>', ' ', regex=False)
reviews['comments'] = reviews['comments'].str.replace('<[^>]+>', ' ', regex=True)

# merge with listings
merged = reviews.merge(
    listings_clean[['id', 'instant_bookable', 'neighbourhood_group_cleansed',
                    'room_type', 'estimated_occupancy_l365d', 'price']],
    left_on='listing_id',
    right_on='id',
    how='inner'
)

print(f'Reviews after merge: {merged.shape}')
print(f'Unique listings in merged: {merged["listing_id"].nunique()}')

Reviews after merge: (936325, 12)
Unique listings in merged: 23976


In [12]:
# review stats
merged['review_stats'] = merged['comments'].str.len()

print('Review stats:')
print(merged['review_length'].describe())

# reviews by year
merged['date'] = pd.to_datetime(merged['date'])
merged['year'] = merged['date'].dt.year

print('\nReviews by year:')
print(merged['year'].value_counts().sort_index())

Review stats:
count    936325.000000
mean        253.214663
std         251.741871
min           1.000000
25%          86.000000
50%         183.000000
75%         336.000000
max        5921.000000
Name: review_length, dtype: float64

Reviews by year:
year
2009        43
2010       416
2011      1632
2012      3229
2013      5865
2014     11264
2015     23056
2016     39896
2017     54965
2018     78602
2019    101845
2020     39131
2021     80502
2022    137689
2023    157147
2024    124488
2025     76555
Name: count, dtype: int64
